In [1]:
!nvidia-smi

Sat May 23 12:20:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# import dependencies
from IPython.display import display, Javascript, Image
from google.colab.output import eval_js
from base64 import b64decode, b64encode
import cv2
import numpy as np
import PIL
import io
import html
import time

In [3]:

# function to convert the JavaScript object into an OpenCV image
def js_to_image(js_reply):
  """
  Params:
          js_reply: JavaScript object containing image from webcam
  Returns:
          img: OpenCV BGR image
  """
  # decode base64 image
  image_bytes = b64decode(js_reply.split(',')[1])
  # convert bytes to numpy array
  jpg_as_np = np.frombuffer(image_bytes, dtype=np.uint8)
  # decode numpy array into OpenCV BGR image
  img = cv2.imdecode(jpg_as_np, flags=1)

  return img

In [4]:
# function to convert OpenCV Rectangle bounding box image into base64 byte string to be overlayed on video stream
def bbox_to_bytes(bbox_array):
  """
  Params:
          bbox_array: Numpy array (pixels) containing rectangle to overlay on video stream.
  Returns:
        bytes: Base64 image byte string
  """
  # convert array into PIL image
  bbox_PIL = PIL.Image.fromarray(bbox_array, 'RGBA')
  iobuf = io.BytesIO()
  # format bbox into png for return
  bbox_PIL.save(iobuf, format='png')
  # format return string
  bbox_bytes = 'data:image/png;base64,{}'.format((str(b64encode(iobuf.getvalue()), 'utf-8')))

  return bbox_bytes

In [5]:
# JavaScript to properly create our live video stream using our webcam as input
def video_stream():
  js = Javascript('''
    var video;
    var div = null;
    var stream;
    var captureCanvas;
    var imgElement;
    var labelElement;

    var pendingResolve = null;
    var shutdown = false;

    function removeDom() {
       stream.getVideoTracks()[0].stop();
       video.remove();
       div.remove();
       video = null;
       div = null;
       stream = null;
       imgElement = null;
       captureCanvas = null;
       labelElement = null;
    }

    function onAnimationFrame() {
      if (!shutdown) {
        window.requestAnimationFrame(onAnimationFrame);
      }
      if (pendingResolve) {
        var result = "";
        if (!shutdown) {
          captureCanvas.getContext('2d').drawImage(video, 0, 0, 640, 480);
          result = captureCanvas.toDataURL('image/jpeg', 0.8)
        }
        var lp = pendingResolve;
        pendingResolve = null;
        lp(result);
      }
    }

    async function createDom() {
      if (div !== null) {
        return stream;
      }

      div = document.createElement('div');
      div.style.border = '2px solid black';
      div.style.padding = '3px';
      div.style.width = '100%';
      div.style.maxWidth = '600px';
      document.body.appendChild(div);

      const modelOut = document.createElement('div');
      modelOut.innerHTML = "Status:";
      labelElement = document.createElement('span');
      labelElement.innerText = 'No data';
      labelElement.style.fontWeight = 'bold';
      modelOut.appendChild(labelElement);
      div.appendChild(modelOut);

      video = document.createElement('video');
      video.style.display = 'block';
      video.width = div.clientWidth - 6;
      video.setAttribute('playsinline', '');
      video.onclick = () => { shutdown = true; };
      stream = await navigator.mediaDevices.getUserMedia(
          {video: { facingMode: "environment"}});
      div.appendChild(video);

      imgElement = document.createElement('img');
      imgElement.style.position = 'absolute';
      imgElement.style.zIndex = 1;
      imgElement.onclick = () => { shutdown = true; };
      div.appendChild(imgElement);

      const instruction = document.createElement('div');
      instruction.innerHTML =
          '' +
          'When finished, click here or on the video to stop this demo';
      div.appendChild(instruction);
      instruction.onclick = () => { shutdown = true; };

      video.srcObject = stream;
      await video.play();

      captureCanvas = document.createElement('canvas');
      captureCanvas.width = 640; //video.videoWidth;
      captureCanvas.height = 480; //video.videoHeight;
      window.requestAnimationFrame(onAnimationFrame);

      return stream;
    }
    async function stream_frame(label, imgData) {
      if (shutdown) {
        removeDom();
        shutdown = false;
        return '';
      }

      var preCreate = Date.now();
      stream = await createDom();

      var preShow = Date.now();
      if (label != "") {
        labelElement.innerHTML = label;
      }

      if (imgData != "") {
        var videoRect = video.getClientRects()[0];
        imgElement.style.top = videoRect.top + "px";
        imgElement.style.left = videoRect.left + "px";
        imgElement.style.width = videoRect.width + "px";
        imgElement.style.height = videoRect.height + "px";
        imgElement.src = imgData;
      }

      var preCapture = Date.now();
      var result = await new Promise(function(resolve, reject) {
        pendingResolve = resolve;
      });
      shutdown = false;

      return {'create': preShow - preCreate,
              'show': preCapture - preShow,
              'capture': Date.now() - preCapture,
              'img': result};
    }
    ''')

  display(js)

def video_frame(label, bbox):
  data = eval_js('stream_frame("{}", "{}")'.format(label, bbox))
  return data

In [6]:
!git clone https://github.com/ultralytics/yolov5

Cloning into 'yolov5'...
remote: Enumerating objects: 17968, done.
remote: Counting objects: 100% (102/102), done.
remote: Compressing objects: 100% (72/72), done.
remote: Total 17968 (delta 78), reused 30 (delta 30), pack-reused 17866 (from 3)
Receiving objects: 100% (17968/17968), 17.11 MiB | 23.23 MiB/s, done.
Resolving deltas: 100% (12222/12222), done.


In [7]:
import os
import sys

In [8]:
!ls yolov5

benchmarks.py	 data	     LICENSE	     README.zh-CN.md   train.py
CITATION.cff	 detect.py   models	     requirements.txt  tutorial.ipynb
classify	 export.py   pyproject.toml  segment	       utils
CONTRIBUTING.md  hubconf.py  README.md	     tests	       val.py


In [9]:
!cd yolov5

In [10]:
!pip install ultralytics


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 22.3 MB/s eta 0:00:00


In [11]:
import torch
model = torch.hub.load('/content/yolov5', 'custom', path='/content/yolov5s.pt', source='local')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
requirements: Ultralytics requirement ['urllib3>=2.6.0 ; python_version > "3.8"'] not found, attempting AutoUpdate...
WARNING ⚠️ Retry 1/2 failed: Command 'uv pip install --no-cache-dir --python "/usr/bin/python3" "urllib3>=2.6.0 ; python_version > "3.8""  --index-strategy=unsafe-best-match --break-system-packages' returned non-zero exit status 2.
WARNING ⚠️ Retry 2/2 failed: Command 'uv pip install --no-cache-dir --python "/usr/bin/python3" "urllib3>=2.6.0 ; python_version > "3.8""  --index-strategy=unsafe-best-match --break-system-packages' returned non-zero exit status 2.
WARNING ⚠️ requirements: ❌ Command 'uv pip install --no-cache-dir --python "/usr/bin/python3" "urllib3>=2.6

YOLOv5 🚀 v7.0-484-g70b964b6 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)

100%|██████████| 14.1M/14.1M [00:00<00:00, 146MB/s]

Fusing layers... 
YOLOv5s summary: 213 layers, 7225885 parameters, 0 gradients, 16.4 GFLOPs
Adding AutoShape... 


In [14]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')
import cv2
from google.colab.patches import cv2_imshow


In [46]:
video_stream()
label_html = 'Capturing...'
bbox = ''
count = 0
while True:
    js_reply = video_frame(label_html, bbox)
    if not js_reply:
        break
    img = js_to_image(js_reply["img"])
    bbox_array = np.zeros([480,640,4], dtype=np.uint8)
    detections = model(img[..., ::-1])
    results = detections.pandas().xyxy[0].to_dict(orient="records")
    for result in results:
      con = result['confidence']
      cs  =str( result['name'])
      x1  = int(result['xmin'])
      y1  = int(result['ymin'])
      x2  = int(result['xmax'])
      y2  = int(result['ymax'])

      cv2.rectangle(bbox_array,(x1, y1), (x2, y2),(255,0,0),2)
      cv2.putText(bbox_array, "{} [{:.2f}]".format(cs, float(con)),
                        (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5,
                        (255,0,0),2)

    bbox_array[:,:,3] = (bbox_array.max(axis = 2) > 0 ).astype(int) * 255
    bbox_bytes = bbox_to_bytes(bbox_array)
    bbox = bbox_bytes

<IPython.core.display.Javascript object>

In [19]:
!pip install pyserial requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 3.9 MB/s eta 0:00:00


In [44]:
import time
import IPython
from google.colab import output

# ==========================================
# 1. INISIALISASI JAVASCRIPT & UI (Wajib ada)
# ==========================================
js_init_code = """
<div style="padding: 10px; border: 1px solid #ccc; background-color: #f9f9f9; border-radius: 5px;">
    <button id="connectBtn" style="padding: 10px; font-weight: bold; background-color: #008CBA; color: white; border: none; cursor: pointer; border-radius: 4px;">
        🔌 Hubungkan ke Arduino
    </button>
    <p id="status" style="margin-top: 10px; color: gray; font-weight: bold;">Status: Terputus</p>
</div>

<script>
let port;
let writer;

// Daftarkan fungsi ke objek window browser
window.sendSerialData = async function(data) {
    if (writer) {
        const encoder = new TextEncoder();
        await writer.write(encoder.encode(data));
        console.log("Data dikirim: " + data);
    } else {
        alert("PENTING: Klik tombol Hubungkan terlebih dahulu!");
    }
}

document.getElementById('connectBtn').addEventListener('click', async () => {
    const statusText = document.getElementById('status');
    try {
        port = await navigator.serial.requestPort();
        await port.open({ baudRate: 9600 });

        const textEncoder = new TextEncoderStream();
        const writableStreamClosed = textEncoder.readable.pipeTo(port.writable);
        writer = textEncoder.writable.getWriter();

        statusText.innerText = "Status: Terhubung & Siap Kirim Data!";
        statusText.style.color = "green";
    } catch (error) {
        statusText.innerText = "Error: " + error.message;
        statusText.style.color = "red";
    }
});
</script>
"""

# Tampilkan tombol koneksi di output
IPython.display.display(IPython.display.HTML(js_init_code))

# ==========================================
# 2. FUNGSI PYTHON UNTUK KONTROL LED
# ==========================================
def kontrol_led(perintah):
    if perintah in ['1', '0']:
        # Ambil fungsi JavaScript yang didaftarkan di atas
        output.eval_js(f"window.sendSerialData('{perintah}')")
        print(f"[Python] Perintah '{perintah}' terkirim.")
    else:
        print("Gunakan '1' atau '0'.")

print("\n--- SIAP ---")
print("1. Silakan klik tombol 'Hubungkan ke Arduino' di atas.")
print("2. Setelah terhubung, buat cell baru di bawah untuk memanggil kontrol_led('1').")


# Membuat tombol ON/OFF interaktif memanfaatkan fungsi yang sudah dibuat
js_buttons = """
<div style="margin-top: 15px;">
    <button onclick="window.sendSerialData('1')" style="padding: 10px 20px; background-color: #4CAF50; color: white; border: none; font-weight: bold; margin-right: 10px; cursor: pointer; border-radius: 4px;">💡 LED ON</button>
    <button onclick="window.sendSerialData('0')" style="padding: 10px 20px; background-color: #f44336; color: white; border: none; font-weight: bold; cursor: pointer; border-radius: 4px;">🔌 LED OFF</button>
</div>
"""
IPython.display.display(IPython.display.HTML(js_buttons))



--- SIAP ---
1. Silakan klik tombol 'Hubungkan ke Arduino' di atas.
2. Setelah terhubung, buat cell baru di bawah untuk memanggil kontrol_led('1').


In [43]:
# Jalankan cell ini untuk membuat LED berkedip 3 kali
import time

print("Memulai tes LED berkedip...")

for i in range(3):
    print(f"Kedipan ke-{i+1}: LED NYALA")
    kontrol_led('1')  # Mengirim perintah '1' ke Arduino
    time.sleep(1)     # Delay 1 detik

    print(f"Kedipan ke-{i+1}: LED MATI")
    kontrol_led('0')  # Mengirim perintah '0' ke Arduino
    time.sleep(1)     # Delay 1 detik

print("Tes selesai!")


Memulai tes LED berkedip...
Kedipan ke-1: LED NYALA


MessageError: TypeError: window.sendSerialData is not a function

In [48]:
import time
import IPython
from google.colab import output
import numpy as np
import cv2

# ==========================================
# 1. INISIALISASI JAVASCRIPT & UI KONEKSI ARDUINO
# ==========================================
js_init_code = """
<div style="padding: 10px; border: 1px solid #ccc; background-color: #f9f9f9; border-radius: 5px; margin-bottom: 10px;">
    <button id="connectBtn" style="padding: 10px; font-weight: bold; background-color: #008CBA; color: white; border: none; cursor: pointer; border-radius: 4px;">
        🔌 Hubungkan ke Arduino
    </button>
    <p id="status" style="margin-top: 10px; color: gray; font-weight: bold;">Status: Terputus (Klik tombol di atas sebelum memulai kamera!)</p>
</div>

<script>
let port;
let writer;

// Fungsi global untuk kirim data via serial browser
window.sendSerialData = async function(data) {
    if (writer) {
        const encoder = new TextEncoder();
        await writer.write(encoder.encode(data));
    }
}

document.getElementById('connectBtn').addEventListener('click', async () => {
    const statusText = document.getElementById('status');
    try {
        port = await navigator.serial.requestPort();
        await port.open({ baudRate: 9600 });

        const textEncoder = new TextEncoderStream();
        const writableStreamClosed = textEncoder.readable.pipeTo(port.writable);
        writer = textEncoder.writable.getWriter();

        statusText.innerText = "Status: Terhubung & Siap Deteksi Objek!";
        statusText.style.color = "green";
    } catch (error) {
        statusText.innerText = "Error: " + error.message;
        statusText.style.color = "red";
    }
});
</script>
"""

# Tampilkan UI tombol koneksi serial terlebih dahulu
IPython.display.display(IPython.display.HTML(js_init_code))

def kontrol_led(perintah):
    try:
        output.eval_js(f"window.sendSerialData('{perintah}')")
    except Exception as e:
        # Menghindari crash jika js belum siap saat dipanggil
        pass

# ==========================================
# 2. PROSES STREAM KAMERA & DETEKSI MANUSIA
# ==========================================

# Jalankan fungsi inisialisasi stream kamera bawaan Colab Anda
video_stream()
label_html = 'Deteksi Objek + Kontrol LED'
bbox = ''

print("\n=== SISTEM SIAP ===")
print("Langkah: Klik 'Hubungkan ke Arduino' di atas, lalu tunggu kamera menyala.\n")

# Variabel untuk mencatat status LED terakhir agar tidak mengirim data terus-menerus
led_sekarang = '0'

while True:
    js_reply = video_frame(label_html, bbox)
    if not js_reply:
        break

    img = js_to_image(js_reply["img"])
    bbox_array = np.zeros([480, 640, 4], dtype=np.uint8)

    # Jalankan model deteksi (YOLOv5)
    detections = model(img[..., ::-1])
    results = detections.pandas().xyxy[0].to_dict(orient="records")

    # Flag penanda apakah manusia ditemukan pada frame ini
    manusia_terdeteksi = False

    for result in results:
        con = result['confidence']
        cs  = str(result['name'])
        x1  = int(result['xmin'])
        y1  = int(result['ymin'])
        x2  = int(result['xmax'])
        y2  = int(result['ymax'])

        # Gambar kotak pembatas objek
        cv2.rectangle(bbox_array, (x1, y1), (x2, y2), (255, 0, 0), 2)
        cv2.putText(bbox_array, "{} [{:.2f}]".format(cs, float(con)),
                    (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5,
                    (255, 0, 0), 2)

        # CEK: Apakah objek tersebut adalah manusia?
        if cs.lower() == 'person':
            manusia_terdeteksi = True

    # ==========================================
    # LOGIKA KONTROL LED BERDASARKAN HASIL DETEKSI
    # ==========================================
    if manusia_terdeteksi:
        if led_sekarang != '1': # Kirim data hanya jika status berubah (hemat bandwidth)
            kontrol_led('1')
            led_sekarang = '1'
            print("[SISTEM] Manusia Terdeteksi -> LED NYALA")
    else:
        if led_sekarang != '0':
            kontrol_led('0')
            led_sekarang = '0'
            print("[SISTEM] Area Bersih -> LED MATI")
    # ==========================================

    bbox_array[:,:,3] = (bbox_array.max(axis = 2) > 0).astype(int) * 255
    bbox_bytes = bbox_to_bytes(bbox_array)
    bbox = bbox_bytes


<IPython.core.display.Javascript object>


=== SISTEM SIAP ===
Langkah: Klik 'Hubungkan ke Arduino' di atas, lalu tunggu kamera menyala.

[SISTEM] Manusia Terdeteksi -> LED NYALA
[SISTEM] Area Bersih -> LED MATI
[SISTEM] Manusia Terdeteksi -> LED NYALA
[SISTEM] Area Bersih -> LED MATI
[SISTEM] Manusia Terdeteksi -> LED NYALA
[SISTEM] Area Bersih -> LED MATI
[SISTEM] Manusia Terdeteksi -> LED NYALA
[SISTEM] Area Bersih -> LED MATI
[SISTEM] Manusia Terdeteksi -> LED NYALA
[SISTEM] Area Bersih -> LED MATI
[SISTEM] Manusia Terdeteksi -> LED NYALA
[SISTEM] Area Bersih -> LED MATI
[SISTEM] Manusia Terdeteksi -> LED NYALA
[SISTEM] Area Bersih -> LED MATI
[SISTEM] Manusia Terdeteksi -> LED NYALA
[SISTEM] Area Bersih -> LED MATI
[SISTEM] Manusia Terdeteksi -> LED NYALA
[SISTEM] Area Bersih -> LED MATI
[SISTEM] Manusia Terdeteksi -> LED NYALA
